# Loading and Running a Pre-trained LLM

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [1]:
import jax
import flax.nnx as nnx
from pathlib import Path

from helper import MiniGPT, generate_story

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> file:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.</p>

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

<p> 📒 &nbsp; For more help, please see the <em>"Appendix – Tips, Help, and Download"</em> Lesson.</p>
</div>

In [2]:
model = MiniGPT()

In [3]:
model

MiniGPT( # Param: 20,212,608 (80.9 MB)
  maxlen=128,
  embedding=TokenAndPositionEmbedding( # Param: 9,673,920 (38.7 MB)
    token_emb=Embed( # Param: 9,649,344 (38.6 MB)
      embedding=Param( # 9,649,344 (38.6 MB)
        value=Array(shape=(50257, 192), dtype=dtype('float32'))
      ),
      num_embeddings=50257,
      features=192,
      dtype=dtype('float32'),
      param_dtype=float32,
      embedding_init=<function variance_scaling.<locals>.init at 0x7fc7565d5bc0>,
      promote_dtype=<function promote_dtype at 0x7fc7562a7ba0>
    ),
    pos_emb=Embed( # Param: 24,576 (98.3 KB)
      embedding=Param( # 24,576 (98.3 KB)
        value=Array(shape=(128, 192), dtype=dtype('float32'))
      ),
      num_embeddings=128,
      features=192,
      dtype=dtype('float32'),
      param_dtype=float32,
      embedding_init=<function variance_scaling.<locals>.init at 0x7fc7565d5bc0>,
      promote_dtype=<function promote_dtype at 0x7fc7562a7ba0>
    )
  ),
  transformer_blocks=[TransformerBloc

## Load the saved checkpoint

In [4]:
import orbax
from orbax import checkpoint

from jax.sharding import SingleDeviceSharding 

In [5]:
cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)

In [6]:
restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)

In [7]:
nnx.state(model)

State({
  'embedding': {
    'pos_emb': {
      'embedding': VariableState( # 24,576 (98.3 KB)
        type=Param,
        value=Array([[-0.17626905, -0.14691259,  0.01483388, ...,  0.00622122,
                 0.07315764,  0.05358156],
               [ 0.07633849,  0.02222663, -0.05893598, ...,  0.01625793,
                 0.00844557,  0.0301149 ],
               [ 0.01113019, -0.09743536, -0.09331388, ..., -0.0538105 ,
                 0.05495994,  0.00899045],
               ...,
               [-0.01381622,  0.0994448 , -0.10009246, ..., -0.06279048,
                -0.14022383,  0.03157161],
               [ 0.04184348,  0.11760371,  0.06157182, ...,  0.00250992,
                 0.09520404, -0.02263948],
               [ 0.00363669,  0.04417814,  0.03465553, ..., -0.02581214,
                 0.06426854, -0.00050747]], dtype=float32)
      )
    },
    'token_emb': {
      'embedding': VariableState( # 9,649,344 (38.6 MB)
        type=Param,
        value=Array([[ 0.07245848, -0

In [8]:
checkpoint_path = Path.cwd() / "model_checkpoint.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()

In [9]:
restored_state = checkpointer.restore(
    checkpoint_path,
    item=nnx.state(model),
    restore_args=restore_args)

nnx.update(model,restored_state)

## Run inference

In [10]:
def create_story(story_prompt, temperature, max_new_tokens):
    return generate_story(model, story_prompt, temperature, max_new_tokens)

In [11]:
create_story("Once upon a time a big bear ", 0.2, 30)

'Once upon a time a big bear ops were in the forest. He was very happy and he was always looking for something to do. One day, he saw a big, shiny rock'

## Interactive chat demo

In [12]:
import gradio as gr

demo = gr.Interface(
    fn=create_story,
    inputs=[
        gr.Textbox(label="Story Prompt"),         
        gr.Slider(
            minimum=0, maximum=1.0, value=0.8, step=0.01, label="Temperature"
        ),
        gr.Slider(minimum=0, maximum=200, value=10, step=1, label="Max Tokens"
        )
    ],
    outputs=["text"]
)

demo.launch(share=True)

* Running on local URL:  https://0.0.0.0:7860
* Running on public URL: https://7ab11005de451e263e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
